In [0]:
# Orchestration: run all remaining Bronze loaders in sequence.
# bronze_cod_population already done; bronze_fieldmaps_boundaries deferred.

import time

BRONZE_NOTEBOOK_DIR = (
    "/Workspace/Users/kmoore2@andrew.cmu.edu/"
    "geo-insight-overlooked-crises/notebooks/bronze"
)

LOADERS = [
    "bronze_hno",
    "bronze_hrp",
    "bronze_fts_plan",
    "bronze_fts_cluster",
    "bronze_fts_globalcluster",
    "bronze_fts_flows",
    "bronze_inform_severity",
    "bronze_cod_population_admin2",
    "bronze_cbpf_allocations",
    "bronze_cbpf_contributions",
    "bronze_cbpf_projects",
    "bronze_cerf_allocations",
    "bronze_acled_events",
    "bronze_acled_severity",
    "bronze_echo_fca",
    "bronze_nrc_neglected",
    "bronze_reliefweb_situation_reports",
]

results = []
for loader in LOADERS:
    notebook_path = f"{BRONZE_NOTEBOOK_DIR}/{loader}"
    start = time.time()
    try:
        run_output = dbutils.notebook.run(notebook_path, timeout_seconds=1800)
        elapsed = time.time() - start
        results.append({
            "loader": loader,
            "status": "success",
            "elapsed_s": round(elapsed, 1),
        })
        print(f"✓ {loader} ({elapsed:.1f}s)")
    except Exception as e:
        elapsed = time.time() - start
        results.append({
            "loader": loader,
            "status": "failed",
            "elapsed_s": round(elapsed, 1),
            "error": str(e)[:500],
        })
        print(f"✗ {loader} FAILED ({elapsed:.1f}s): {str(e)[:200]}")
        # Continue rather than stop at the first failure

# COMMAND ----------

# Summary
import pandas as pd
results_df = pd.DataFrame(results)
print(f"\n=== Bronze run summary ===")
print(f"Total: {len(results)}")
print(f"Success: {(results_df['status'] == 'success').sum()}")
print(f"Failed: {(results_df['status'] == 'failed').sum()}")
print(f"Total elapsed: {results_df['elapsed_s'].sum():.1f}s")
display(results_df)

In [0]:
bronze_tables = [
    "bronze_acled_events", "bronze_acled_severity",
    "bronze_cbpf_allocations", "bronze_cbpf_contributions", "bronze_cbpf_projects",
    "bronze_cerf_allocations",
    "bronze_cod_population", "bronze_cod_population_admin2",
    "bronze_echo_fca",
    "bronze_fts_cluster", "bronze_fts_flows", "bronze_fts_globalcluster", "bronze_fts_plan",
    "bronze_hno", "bronze_hrp",
    "bronze_inform_severity",
    "bronze_nrc_neglected",
    "bronze_reliefweb_situation_reports",
]
print(f"{'table':<45} {'rows':>12}")
print("-" * 60)
total = 0
for t in bronze_tables:
    n = spark.table(f"geo_insight.bronze.{t}").count()
    total += n
    print(f"{t:<45} {n:>12,}")
print("-" * 60)
print(f"{'TOTAL':<45} {total:>12,}")